In [ ]:
from pathlib import Path
import html
import os

import yaml
from IPython.display import HTML, display


def load_qa_config(config_path="../configs/automatic_photometric_qa.yaml"):
    """Load the release QA configuration file."""

    with open(config_path, encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    if not isinstance(config, dict):
        raise ValueError("The release QA configuration must be a YAML mapping.")

    return config


def get_config_section(config, section_name):
    """Return a required configuration section."""

    section = config.get(section_name)

    if not isinstance(section, dict):
        raise ValueError(f"Missing or invalid configuration section: {section_name}")

    return section


config_path = Path(os.environ.get("AUTOMATIC_PHOTOMETRIC_QA_CONFIG", "../configs/automatic_photometric_qa.yaml")).expanduser().resolve()
config_dir = config_path.parent
config = load_qa_config(config_path)

notebook_config = get_config_section(config, "notebook")
cluster_config = get_config_section(config, "cluster")

if "catalogs" in config:
    catalog_configs = config["catalogs"]

    if not isinstance(catalog_configs, list) or not catalog_configs:
        raise ValueError("catalogs must be a non-empty list of catalog configurations.")
else:
    catalog_config = get_config_section(config, "catalog")
    catalog_configs = [
        {
            **catalog_config,
            "title": catalog_config.get("title", notebook_config["title"]),
            "basic_statistics": config.get("basic_statistics"),
            "unique_count": config.get("unique_count"),
            "spatial_distribution": config.get("spatial_distribution"),
            "survey_area": config.get("survey_area"),
            "magnitudes": config.get("magnitudes"),
            "magnitude_errors": config.get("magnitude_errors"),
            "magnitude_error_trends": config.get("magnitude_error_trends"),
        }
    ]

for catalog_index, catalog_config in enumerate(catalog_configs, start=1):
    if not isinstance(catalog_config, dict):
        raise ValueError(f"catalogs[{catalog_index}] must be a YAML mapping.")

    status = catalog_config.get("status", "available")

    if status not in {"available", "planned"}:
        raise ValueError(
            f"catalogs[{catalog_index}].status must be either 'available' "
            f"or 'planned', got: {status!r}"
        )

    omit_paths = catalog_config.get("omit_paths", False)

    if not isinstance(omit_paths, bool):
        raise ValueError(
            f"catalogs[{catalog_index}].omit_paths must be true or false, "
            f"got: {omit_paths!r}"
        )

    if status == "available" and "path" not in catalog_config:
        raise ValueError(
            f"catalogs[{catalog_index}] is available and is missing required key: "
            "path. Use status: planned for placeholder catalog sections without "
            "data files."
        )

    catalog_config.setdefault("title", f"Catalog {catalog_index}")

notebook_title = notebook_config["title"]
notebook_subtitle = notebook_config.get("subtitle", "")
last_verified_run = notebook_config.get("last_verified_run")


def resolve_config_path(path_value):
    """Resolve a configured path relative to the YAML file."""

    candidate = Path(path_value).expanduser()

    if candidate.is_absolute():
        return candidate

    return (config_dir / candidate).resolve()


In [ ]:
logo_html = """
<div style="display: flex; align-items: center; gap: 24px; margin-bottom: 18px;">
  <div style="display: flex; align-items: center; gap: 18px; flex: 0 0 auto;">
    <img src="https://www.linea.org.br/brand/linea-logo-color.svg" width="100">
    <img src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width="180">
  </div>
  <div style="min-width: 0;">
    <h1 style="margin: 0 0 8px 0;">{title}</h1>
    {subtitle}
    {last_verified_run}
  </div>
</div>
"""

subtitle_html = ""

if notebook_subtitle:
    subtitle_html = (
        '<p style="font-size: 18px; margin: 0 0 6px 0;">'
        f"{html.escape(notebook_subtitle)}"
        "</p>"
    )

last_verified_run_html = ""

if last_verified_run:
    last_verified_run_html = (
        '<p style="margin: 0;">Last verified run: '
        f"<b>{html.escape(str(last_verified_run))}</b></p>"
    )

display(
    HTML(
        logo_html.format(
            title=html.escape(notebook_title),
            subtitle=subtitle_html,
            last_verified_run=last_verified_run_html,
        )
    )
)


In [ ]:
notebook_introduction = notebook_config.get("introduction")

if notebook_introduction:
    display(HTML(f"""
    <hr>
    <p>{html.escape(str(notebook_introduction))}</p>
    <hr>
    """))


In [ ]:
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/automatic-photometric-qa-matplotlib")

import dask
import dask.array as da
import dask.dataframe as dd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dask import delayed
from dask.distributed import Client, LocalCluster
from IPython.display import Markdown
from matplotlib.colors import LogNorm

try:
    from dask_jobqueue import SLURMCluster
except ImportError:
    SLURMCluster = None

warnings.filterwarnings(
    "ignore",
    message="Sending large graph of size.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in subtract",
    category=RuntimeWarning,
    module=r"dask\.array\.reductions",
)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in subtract",
    category=RuntimeWarning,
    module=r"numpy\.lib\._function_base_impl",
)
warnings.filterwarnings(
    "ignore",
    message="IProgress not found. Please update jupyter and ipywidgets.*",
    category=Warning,
    module=r"tqdm\.auto",
)

expected_warning_filters = [
    "ignore:invalid value encountered in subtract:RuntimeWarning:dask.array.reductions",
    "ignore:invalid value encountered in subtract:RuntimeWarning:numpy.lib._function_base_impl",
    "ignore:IProgress not found. Please update jupyter and ipywidgets.*:Warning:tqdm.auto",
]
existing_python_warnings = os.environ.get("PYTHONWARNINGS")
os.environ["PYTHONWARNINGS"] = ",".join(
    [
        *([existing_python_warnings] if existing_python_warnings else []),
        *expected_warning_filters,
    ]
)

sns.set_theme(style="whitegrid")


QA_PROGRESS_PREFIX = "QA_PROGRESS:"


def qa_log(message):
    """Emit a progress line for the command-line runner."""

    print(f"{QA_PROGRESS_PREFIX} {message}", flush=True)


def get_basic_statistics_columns(catalog_columns, basic_statistics_config):
    """Resolve the configured basic-statistics column selection."""

    available_columns = list(catalog_columns)
    selection = basic_statistics_config.get("columns")
    default_first_n = int(basic_statistics_config.get("default_first_n", 20))
    max_columns = int(basic_statistics_config.get("max_columns", 100))

    if selection is None:
        return available_columns[:default_first_n]

    if isinstance(selection, str):
        if selection.lower() == "all":
            if not basic_statistics_config.get("allow_all_columns", False):
                raise ValueError(
                    'basic_statistics.columns is set to "all", which can be expensive '
                    "for wide catalogs. Set basic_statistics.allow_all_columns: true "
                    "to confirm that all columns should be processed."
                )

            warnings.warn(
                'basic_statistics.columns="all" was explicitly allowed. All catalog '
                "columns will be processed, which can be expensive for wide catalogs.",
                UserWarning,
            )

            return available_columns

        raise ValueError(
            'basic_statistics.columns must be null, "all", or a list of column names.'
        )

    if isinstance(selection, list):
        missing_columns = [column for column in selection if column not in available_columns]

        if missing_columns:
            raise ValueError(
                "Configured basic-statistics columns are not present in the catalog: "
                + ", ".join(missing_columns)
            )

        if len(selection) > max_columns and not basic_statistics_config.get("allow_many_columns", False):
            raise ValueError(
                f"basic_statistics.columns contains {len(selection)} columns, which exceeds "
                f"basic_statistics.max_columns={max_columns}. This can create large Dask "
                "graphs and heavy reductions. Increase max_columns or set "
                "basic_statistics.allow_many_columns: true to confirm this choice."
            )

        if len(selection) > max_columns:
            warnings.warn(
                f"basic_statistics.columns contains {len(selection)} columns, exceeding "
                f"basic_statistics.max_columns={max_columns}, but allow_many_columns is true. "
                "This run may create large Dask graphs and heavy reductions.",
                UserWarning,
            )

        return selection

    raise ValueError(
        'basic_statistics.columns must be null, "all", or a list of column names.'
    )


def make_dask_cluster(cluster_config):
    """Create a Dask client and cluster from the configured backend."""

    cluster_type = cluster_config.get("type", "local").lower()

    if cluster_type == "local":
        local_config = get_config_section(cluster_config, "local")
        cluster = LocalCluster(
            n_workers=int(local_config.get("n_workers", 3)),
            threads_per_worker=int(local_config.get("cores", 2)),
            memory_limit=local_config.get("memory", "2GB"),
        )
        client = Client(cluster)
        return client, cluster

    if cluster_type == "slurm":
        if SLURMCluster is None:
            raise ImportError(
                "dask_jobqueue is required when cluster.type is set to 'slurm'."
            )

        slurm_config = get_config_section(cluster_config, "slurm")
        cluster = SLURMCluster(
            n_workers=int(slurm_config["n_workers"]),
            queue=slurm_config["queue"],
            account=slurm_config["account"],
            interface=slurm_config["interface"],
            cores=int(slurm_config["cores"]),
            processes=int(slurm_config["processes"]),
            memory=slurm_config["memory"],
            walltime=slurm_config["walltime"],
        )

        adapt_config = slurm_config.get("adapt")

        if adapt_config:
            cluster.adapt(
                minimum_jobs=int(adapt_config["minimum_jobs"]),
                maximum_jobs=int(adapt_config["maximum_jobs"]),
            )

        client = Client(cluster)
        wait_for_workers = slurm_config.get("wait_for_workers")

        if wait_for_workers:
            client.wait_for_workers(int(wait_for_workers))

        return client, cluster

    raise ValueError("cluster.type must be either 'local' or 'slurm'.")


def make_band_model_columns(bands, models):
    """Build band/model column names."""

    return [
        f"{band}_{model}"
        for band in bands
        for model in models
    ]




def model_uses_flux_conversion(model):
    """Return True when a configured model should be converted from flux."""

    return "Flux" in model


def model_uses_flux_error_conversion(model):
    """Return True when a configured error model should be converted from flux error."""

    return "FluxErr" in model


def get_mag_offset(section_config):
    """Return the configured magnitude zero-point offset."""

    mag_offset = section_config.get("mag_offset")

    if mag_offset is None:
        raise ValueError(
            "mag_offset is required when converting Flux columns to magnitudes."
        )

    return float(mag_offset)


def _convert_magnitude_partition(partition, bands, models, mag_offset):
    """Convert flux columns in one LSDB partition, preserving its index."""

    partition = partition.copy()
    for band in bands:
        for model in models:
            if model_uses_flux_conversion(model):
                column = f"{band}_{model}"
                flux = pd.to_numeric(partition[column], errors="coerce").astype("float64")
                valid = (flux > 0) & np.isfinite(flux)
                partition[column] = mag_offset - 2.5 * np.log10(flux.where(valid))
    return partition


def _convert_error_partition(partition, bands, models):
    """Convert flux errors to magnitude errors in one LSDB partition."""

    partition = partition.copy()
    conversion_factor = 2.5 / np.log(10.0)
    for band in bands:
        for model in models:
            if model_uses_flux_error_conversion(model):
                error_column = f"{band}_{model}"
                flux_column = f"{band}_{flux_error_model_to_flux_model(model)}"
                flux = pd.to_numeric(partition[flux_column], errors="coerce").astype("float64")
                error = pd.to_numeric(partition[error_column], errors="coerce").astype("float64")
                valid = (flux > 0) & (error >= 0) & np.isfinite(flux) & np.isfinite(error)
                partition[error_column] = (conversion_factor * error / flux).where(valid)
    return partition


def make_flux_magnitude_source(catalog_context, bands, models, section_config):
    """
    Load magnitude inputs and lazily convert Flux columns to magnitudes.

    Non-positive, missing, and non-finite flux values become NaN through the
    lazy Dask expression and are excluded later by the finite-value filters used
    in histograms and distribution statistics.
    """

    output_columns = make_band_model_columns(bands, models)
    conversion_models = [model for model in models if model_uses_flux_conversion(model)]
    mag_offset = get_mag_offset(section_config) if conversion_models else None

    raw_columns = sorted(set(output_columns))
    source = read_catalog_columns(catalog_context, columns=raw_columns)

    if catalog_context["kind"] == "hats":
        if conversion_models:
            transform = lambda partition: _convert_magnitude_partition(
                partition, bands, models, mag_offset
            )
            source = source.map_partitions(transform, meta=transform(source.meta))
        return source[output_columns], output_columns

    for band in bands:
        for model in conversion_models:
            column = f"{band}_{model}"
            finite_positive_flux = source[column].where(
                (source[column] > 0) & da.isfinite(source[column])
            )
            source[column] = mag_offset - 2.5 * np.log10(finite_positive_flux)

    return source[output_columns], output_columns


def flux_error_model_to_flux_model(error_model):
    """Return the matching flux model for a FluxErr model name."""

    if error_model.endswith("FluxErr"):
        return error_model[: -len("Err")]

    raise ValueError(
        "Flux error conversion requires model names ending in 'FluxErr'."
    )


def make_magnitude_error_source(catalog_context, bands, models, section_config):
    """Load error inputs and lazily convert FluxErr columns to magnitude errors."""

    output_columns = make_band_model_columns(bands, models)
    conversion_models = [
        model for model in models if model_uses_flux_error_conversion(model)
    ]

    raw_columns = set(output_columns)

    for model in conversion_models:
        flux_model = flux_error_model_to_flux_model(model)
        raw_columns.update(f"{band}_{flux_model}" for band in bands)

    source = read_catalog_columns(catalog_context, columns=sorted(raw_columns))

    if catalog_context["kind"] == "hats":
        if conversion_models:
            transform = lambda partition: _convert_error_partition(partition, bands, models)
            source = source.map_partitions(transform, meta=transform(source.meta))
        return source[output_columns], output_columns

    conversion_factor = 2.5 / np.log(10.0)

    for band in bands:
        for model in conversion_models:
            error_column = f"{band}_{model}"
            flux_column = f"{band}_{flux_error_model_to_flux_model(model)}"
            valid = (
                (source[flux_column] > 0)
                & (source[error_column] >= 0)
                & da.isfinite(source[flux_column])
                & da.isfinite(source[error_column])
            )
            converted_error = conversion_factor * source[error_column] / source[flux_column]
            source[error_column] = converted_error.where(valid)

    return source[output_columns], output_columns


def make_magnitude_error_trend_source(
    catalog_context, bands, magnitude_models, error_models, magnitudes_config
):
    """Derive paired magnitude and error columns in one LSDB Catalog."""

    magnitude_columns = make_band_model_columns(bands, magnitude_models)
    error_columns = make_band_model_columns(bands, error_models)
    raw_columns = set(magnitude_columns + error_columns)
    for model in error_models:
        if model_uses_flux_error_conversion(model):
            raw_columns.update(
                f"{band}_{flux_error_model_to_flux_model(model)}" for band in bands
            )
    source = read_catalog_columns(catalog_context, columns=sorted(raw_columns))
    mag_offset = (
        get_mag_offset(magnitudes_config)
        if any(model_uses_flux_conversion(model) for model in magnitude_models)
        else None
    )

    def transform(partition):
        partition = _convert_error_partition(partition, bands, error_models)
        partition = _convert_magnitude_partition(
            partition, bands, magnitude_models, mag_offset
        )
        return partition[magnitude_columns + error_columns]

    return source.map_partitions(transform, meta=transform(source.meta))

def infer_magnitude_error_model(magnitude_model, magnitude_errors_config):
    """Infer the configured error model that matches a magnitude model."""

    candidate = f"{magnitude_model}Err"
    configured_error_models = set(magnitude_errors_config.get("models", []))

    if candidate in configured_error_models:
        return candidate

    raise ValueError(
        "magnitude_error_trends could not infer the error model for "
        f"'{magnitude_model}'. Configure magnitude_error_trends.error_models "
        "with one entry for each magnitude model."
    )


def get_magnitude_error_trend_models(trend_config, magnitudes_config, magnitude_errors_config):
    """Return magnitude/error model pairs for the trend plot."""

    magnitude_models = trend_config.get("models", magnitudes_config["models"])
    error_models = trend_config.get("error_models")

    if error_models is None:
        error_models = [
            infer_magnitude_error_model(model, magnitude_errors_config)
            for model in magnitude_models
        ]

    if len(magnitude_models) != len(error_models):
        raise ValueError(
            "magnitude_error_trends.models and magnitude_error_trends.error_models "
            "must have the same length."
        )

    return list(magnitude_models), list(error_models)


def _qa_map_partitions(source, function, *args, meta):
    """Run a partition-level function while suppressing known LSDB warnings."""

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="output of the function must be a DataFrame to generate an LSDB.*",
            category=RuntimeWarning,
        )
        return source.map_partitions(function, *args, meta=meta)


# Count and cardinality helpers

def _qa_partition_row_count(partition):
    """Count rows in one partition."""

    return pd.Series([len(partition)], name="count", dtype="int64")


def qa_row_count(source):
    """Compute the total row count for a distributed table."""

    counts = _qa_map_partitions(
        source,
        _qa_partition_row_count,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()

    return int(counts.sum())


def _qa_partition_value_counts(partition, column):
    """Compute value counts for one partition."""

    return partition[column].value_counts(dropna=False).rename("count")


def qa_value_counts(source, column):
    """Compute exact value counts for a distributed column."""

    partials = _qa_map_partitions(
        source,
        _qa_partition_value_counts,
        column,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()

    return partials.groupby(level=0, dropna=False).sum().sort_index()


def _qa_partition_unique_values(partition, column, max_unique_values, dropna):
    """Return at most cap + 1 unique values from one partition."""

    values = partition[column]
    if dropna:
        values = values.dropna()
    values = values.drop_duplicates().head(max_unique_values + 1)

    return pd.Series(values.to_numpy(), name=column)


def _qa_merge_unique_values(left, right, max_unique_values):
    """Merge bounded unique sets while preserving overflow detection."""

    values = pd.concat([left, right], ignore_index=True).drop_duplicates()
    return values.head(max_unique_values + 1).reset_index(drop=True)


def qa_unique_count(source, column, dropna=True, max_unique_values=10000, return_values=False):
    """
    Compute an exact global unique count with a configured cardinality cap.

    The function either returns an exact global count, optionally with the exact
    unique values, or raises an error. It does not return approximate, sampled,
    or per-partition counts.
    """

    if max_unique_values <= 0:
        raise ValueError("max_unique_values must be greater than zero.")

    partial_uniques = _qa_map_partitions(
        source,
        _qa_partition_unique_values,
        column,
        int(max_unique_values),
        dropna,
        meta=pd.Series(name=column, dtype=source[column].dtype),
    ).to_delayed()

    # Reduce on workers. Collecting every partition's values on the driver can
    # be much larger than the final global set for a large HATS catalog.
    while len(partial_uniques) > 1:
        partial_uniques = [
            delayed(_qa_merge_unique_values)(
                partial_uniques[index], partial_uniques[index + 1], max_unique_values
            )
            if index + 1 < len(partial_uniques)
            else partial_uniques[index]
            for index in range(0, len(partial_uniques), 2)
        ]

    unique_values = (
        pd.Index(partial_uniques[0].compute())
        if partial_uniques
        else pd.Index([])
    )

    if len(unique_values) > max_unique_values:
        raise ValueError(
            f"Exact unique count for column '{column}' exceeded "
            f"unique_count.max_unique_values={max_unique_values}. No approximate "
            "or partial value was reported. Increase max_unique_values only if this "
            "high-cardinality exact count is scientifically required and the driver "
            "has enough memory."
        )

    if return_values:
        return int(len(unique_values)), unique_values

    return int(len(unique_values))


# Survey area helpers

def _qa_partition_healpix_pixels(partition, ra_column, dec_column, order):
    """Return unique HEALPix pixels touched by one partition."""

    from hats.pixel_math import healpix_shim

    ra = pd.to_numeric(partition[ra_column], errors="coerce").to_numpy()
    dec = pd.to_numeric(partition[dec_column], errors="coerce").to_numpy()
    valid = np.isfinite(ra) & np.isfinite(dec) & (dec >= -90.0) & (dec <= 90.0)

    if not np.any(valid):
        return pd.Series([], name="healpix_pixel", dtype="int64")

    pixels = healpix_shim.radec2pix(int(order), ra[valid] % 360.0, dec[valid])

    return pd.Series(np.unique(pixels).astype("int64"), name="healpix_pixel")


def _qa_partition_parent_healpix_pixels(partition, healpix_column, source_order, target_order):
    """Return unique target-order parents from a HEALPix-indexed partition."""

    pixels = pd.to_numeric(partition[healpix_column], errors="coerce").to_numpy()
    pixels = pixels[np.isfinite(pixels)].astype("int64", copy=False)

    if pixels.size == 0:
        return pd.Series([], name="healpix_pixel", dtype="int64")

    order_delta = int(source_order) - int(target_order)
    parent_pixels = pixels // (4 ** order_delta)

    return pd.Series(np.unique(parent_pixels), name="healpix_pixel")


def qa_count_unique_healpix_pixels(partition_pixels, split_every=8, split_out=64, shuffle_method=None):
    """Count unique HEALPix pixels with distributed Dask deduplication."""

    unique_pixels = partition_pixels.drop_duplicates(
        split_every=split_every,
        split_out=split_out,
        shuffle_method=shuffle_method,
    )

    return int(unique_pixels.size.compute())


def make_survey_area_result(order, unique_pixel_count):
    """Build scalar survey-area results for a HEALPix order and pixel count."""

    from hats.pixel_math import healpix_shim

    nside = int(healpix_shim.order2nside(order))
    pixel_area_sq_deg = float(healpix_shim.order2pixarea(order, degrees=True))

    return {
        "order": order,
        "nside": nside,
        "unique_pixels": unique_pixel_count,
        "pixel_area_sq_deg": pixel_area_sq_deg,
        "area_sq_deg": unique_pixel_count * pixel_area_sq_deg,
    }


def qa_survey_area(source, ra_column, dec_column, order=12, split_every=8, split_out=64, shuffle_method=None):
    """Estimate survey area from the number of occupied HEALPix pixels."""

    order = int(order)
    split_every = int(split_every)
    split_out = int(split_out)

    if order < 0:
        raise ValueError("survey_area.order must be greater than or equal to zero.")

    if split_every <= 0:
        raise ValueError("survey_area.split_every must be greater than zero.")

    if split_out <= 0:
        raise ValueError("survey_area.split_out must be greater than zero.")

    coordinate_data = source[[ra_column, dec_column]]
    partition_pixels = _qa_map_partitions(
        coordinate_data,
        _qa_partition_healpix_pixels,
        ra_column,
        dec_column,
        order,
        meta=pd.Series(name="healpix_pixel", dtype="int64"),
    )
    unique_pixel_count = qa_count_unique_healpix_pixels(
        partition_pixels,
        split_every=split_every,
        split_out=split_out,
        shuffle_method=shuffle_method,
    )

    return make_survey_area_result(order, unique_pixel_count)


def qa_survey_area_from_healpix(source, healpix_column, source_order, target_order=12, split_every=8, split_out=64, shuffle_method=None):
    """Estimate survey area by degrading an existing HEALPix column."""

    source_order = int(source_order)
    target_order = int(target_order)
    split_every = int(split_every)
    split_out = int(split_out)

    if target_order < 0:
        raise ValueError("survey_area.order must be greater than or equal to zero.")

    if source_order < target_order:
        raise ValueError(
            "survey_area.healpix_column_order must be greater than or equal to "
            "survey_area.order. Use RA/Dec columns for finer target orders."
        )

    if split_every <= 0:
        raise ValueError("survey_area.split_every must be greater than zero.")

    if split_out <= 0:
        raise ValueError("survey_area.split_out must be greater than zero.")

    partition_pixels = _qa_map_partitions(
        source[[healpix_column]],
        _qa_partition_parent_healpix_pixels,
        healpix_column,
        source_order,
        target_order,
        meta=pd.Series(name="healpix_pixel", dtype="int64"),
    )
    unique_pixel_count = qa_count_unique_healpix_pixels(
        partition_pixels,
        split_every=split_every,
        split_out=split_out,
        shuffle_method=shuffle_method,
    )

    return make_survey_area_result(target_order, unique_pixel_count)


# Histogram helpers

def _qa_partition_histogram2d_array(
    partition,
    ra_column,
    dec_column,
    xedges,
    yedges,
):
    """Compute a fixed-size 2D histogram for one catalog partition."""

    ra = pd.to_numeric(partition[ra_column], errors="coerce").to_numpy()
    dec = pd.to_numeric(partition[dec_column], errors="coerce").to_numpy()

    # Astronomical Mollweide convention: RA = 0 deg at the center,
    # and RA increases to the left.
    x = -np.deg2rad(((ra + 180.0) % 360.0) - 180.0)
    y = np.deg2rad(dec)

    valid = np.isfinite(x) & np.isfinite(y) & (dec >= -90.0) & (dec <= 90.0)
    counts, _, _ = np.histogram2d(x[valid], y[valid], bins=[xedges, yedges])

    return counts.astype("int64", copy=False)


def qa_histogram2d(
    source,
    ra_column,
    dec_column,
    xedges,
    yedges,
    split_every=8,
):
    """Compute an exact distributed 2D histogram."""

    output_shape = (len(xedges) - 1, len(yedges) - 1)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histogram2d_array)(
            partition,
            ra_column,
            dec_column,
            xedges,
            yedges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        return np.zeros(output_shape, dtype="int64")

    stacked = da.stack(partition_histograms, axis=0)
    total = stacked.sum(axis=0, dtype="int64", split_every=split_every)

    return total.compute()


def _qa_partition_histograms1d_array(partition, columns, edges):
    """Compute fixed-bin 1D histograms for one catalog partition."""

    partition_counts = np.zeros(
        (len(columns), len(edges) - 1),
        dtype="int64",
    )

    for column_index, column in enumerate(columns):
        values = pd.to_numeric(
            partition[column],
            errors="coerce",
        ).to_numpy()

        values = values[np.isfinite(values)]
        counts, _ = np.histogram(values, bins=edges)
        partition_counts[column_index] = counts

    return partition_counts


def qa_histograms1d(
    source,
    columns,
    bins=50,
    value_range=None,
    split_every=8,
):
    """Compute exact histograms for multiple columns in one distributed pass."""

    if value_range is None:
        raise ValueError(
            "value_range must be provided when computing multiple histograms in one pass."
        )

    edges = np.linspace(value_range[0], value_range[1], bins + 1)
    output_shape = (len(columns), bins)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histograms1d_array)(
            partition,
            columns,
            edges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        empty_counts = {column: np.zeros(bins, dtype="int64") for column in columns}
        return empty_counts, edges

    stacked = da.stack(partition_histograms, axis=0)
    total_counts = stacked.sum(axis=0, dtype="int64", split_every=split_every).compute()

    histograms = {
        column: total_counts[column_index]
        for column_index, column in enumerate(columns)
    }

    return histograms, edges



def _qa_partition_binned_relation_array(partition, column_pairs, x_edges, y_edges):
    """Compute fixed-bin 2D histograms for multiple x/y column pairs."""

    partition_counts = np.zeros(
        (len(column_pairs), len(x_edges) - 1, len(y_edges) - 1),
        dtype="int64",
    )

    for pair_index, (x_column, y_column) in enumerate(column_pairs):
        x_values = pd.to_numeric(
            partition[x_column],
            errors="coerce",
        ).to_numpy()
        y_values = pd.to_numeric(
            partition[y_column],
            errors="coerce",
        ).to_numpy()

        valid = np.isfinite(x_values) & np.isfinite(y_values)
        counts, _, _ = np.histogram2d(
            x_values[valid],
            y_values[valid],
            bins=[x_edges, y_edges],
        )
        partition_counts[pair_index] = counts.astype("int64", copy=False)

    return partition_counts


def qa_binned_relation_histograms(
    source,
    column_pairs,
    x_bins=50,
    y_bins=50,
    x_range=None,
    y_range=None,
    split_every=8,
):
    """Compute exact 2D histograms for multiple binned relations."""

    if x_range is None or y_range is None:
        raise ValueError(
            "x_range and y_range must be provided for binned relation histograms."
        )

    x_edges = np.linspace(x_range[0], x_range[1], x_bins + 1)
    y_edges = np.linspace(y_range[0], y_range[1], y_bins + 1)
    output_shape = (len(column_pairs), x_bins, y_bins)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_binned_relation_array)(
            partition,
            column_pairs,
            x_edges,
            y_edges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        empty_counts = {
            pair: np.zeros((x_bins, y_bins), dtype="int64")
            for pair in column_pairs
        }
        return empty_counts, x_edges, y_edges

    stacked = da.stack(partition_histograms, axis=0)
    total_counts = stacked.sum(axis=0, dtype="int64", split_every=split_every).compute()

    histograms = {
        pair: total_counts[pair_index]
        for pair_index, pair in enumerate(column_pairs)
    }

    return histograms, x_edges, y_edges


def summarize_binned_relation(histogram2d, y_edges, quantiles=(0.16, 0.84)):
    """Return count, binned mean y, and approximate y quantiles for each x bin."""

    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    counts = histogram2d.sum(axis=1)

    weighted_sum = (histogram2d * y_centers).sum(axis=1)
    mean = np.full(histogram2d.shape[0], np.nan, dtype="float64")
    np.divide(weighted_sum, counts, out=mean, where=counts > 0)

    cumulative = np.cumsum(histogram2d, axis=1)
    quantile_values = []

    for quantile in quantiles:
        targets = counts * float(quantile)
        indices = np.argmax(cumulative >= targets[:, None], axis=1)
        values = y_centers[indices].astype("float64", copy=True)
        values[counts == 0] = np.nan
        quantile_values.append(values)

    return counts, mean, quantile_values


# Distribution statistics helpers

def _qa_partition_distribution_summary(
    partition,
    columns,
    value_range,
    histogram_edges,
    thresholds,
):
    """
    Compute histogram and scalar summaries for one catalog partition.

    For each column, the output contains histogram counts, the count and sum
    of valid values, counts below configured thresholds, and diagnostic counts
    for non-finite, below-range, and above-range values.
    """

    number_of_bins = len(histogram_edges) - 1
    number_of_thresholds = len(thresholds)
    number_of_summary_fields = 2 + number_of_thresholds + 3

    partition_summary = np.zeros(
        (
            len(columns),
            number_of_bins + number_of_summary_fields,
        ),
        dtype="float64",
    )

    lower_limit, upper_limit = value_range

    for column_index, column in enumerate(columns):
        values = pd.to_numeric(
            partition[column],
            errors="coerce",
        ).to_numpy()

        finite_mask = np.isfinite(values)
        valid_mask = finite_mask & (values >= lower_limit) & (values <= upper_limit)
        valid_values = values[valid_mask]

        histogram_counts, _ = np.histogram(valid_values, bins=histogram_edges)

        field_index = number_of_bins

        # Histogram counts.
        partition_summary[column_index, :number_of_bins] = histogram_counts

        # Valid count.
        partition_summary[column_index, field_index] = valid_values.size
        field_index += 1

        # Sum of valid values.
        partition_summary[column_index, field_index] = valid_values.sum(dtype="float64")
        field_index += 1

        # Counts below configured thresholds.
        for threshold in thresholds:
            partition_summary[column_index, field_index] = np.count_nonzero(
                valid_values <= threshold
            )
            field_index += 1

        # NaN, +inf, and -inf values.
        partition_summary[column_index, field_index] = np.count_nonzero(~finite_mask)
        field_index += 1

        # Finite values below the selected range.
        partition_summary[column_index, field_index] = np.count_nonzero(
            finite_mask & (values < lower_limit)
        )
        field_index += 1

        # Finite values above the selected range.
        partition_summary[column_index, field_index] = np.count_nonzero(
            finite_mask & (values > upper_limit)
        )

    return partition_summary


def qa_histogram_quantiles(histogram_counts, histogram_edges, quantiles):
    """Estimate quantiles by interpolating within fixed histogram bins."""

    result = np.full((len(histogram_counts), len(quantiles)), np.nan, dtype="float64")
    for column_index, bin_counts in enumerate(histogram_counts):
        cumulative = np.cumsum(bin_counts)
        if cumulative.size == 0 or cumulative[-1] == 0:
            continue
        for quantile_index, quantile in enumerate(quantiles):
            target = float(quantile) * cumulative[-1]
            bin_index = (
                int(np.flatnonzero(bin_counts)[0])
                if target == 0
                else int(np.searchsorted(cumulative, target, side="left"))
            )
            previous_count = cumulative[bin_index - 1] if bin_index else 0
            fraction = (target - previous_count) / bin_counts[bin_index]
            result[column_index, quantile_index] = (
                histogram_edges[bin_index]
                + fraction * (histogram_edges[bin_index + 1] - histogram_edges[bin_index])
            )
    return result


def qa_distribution_statistics(
    source,
    columns,
    value_range,
    peak_bin_width,
    thresholds=(),
    quantiles=(0.16, 0.50, 0.84, 0.95),
    split_every=8,
    quantile_method="dask",
):
    """
    Compute distributed descriptive statistics for multiple columns.

    Count, mean, histogram peak, threshold fractions, and diagnostic counts use
    partition-level reductions. Quantiles use either the same fixed histogram
    or Dask's distributed quantile implementation.
    """

    lower_limit, upper_limit = value_range

    if lower_limit >= upper_limit:
        raise ValueError("value_range must satisfy lower_limit < upper_limit.")

    if peak_bin_width <= 0:
        raise ValueError("peak_bin_width must be greater than zero.")

    thresholds = tuple(thresholds)
    quantiles = tuple(quantiles)

    if quantile_method not in {"dask", "histogram"}:
        raise ValueError("statistics.quantile_method must be 'dask' or 'histogram'.")

    if any(quantile < 0 or quantile > 1 for quantile in quantiles):
        raise ValueError("statistics.quantiles must be between zero and one.")

    number_of_bins = int(np.ceil((upper_limit - lower_limit) / peak_bin_width))

    # This guarantees that the first and last edges match value_range.
    histogram_edges = np.linspace(lower_limit, upper_limit, number_of_bins + 1)

    effective_bin_width = histogram_edges[1] - histogram_edges[0]
    histogram_centers = (histogram_edges[:-1] + histogram_edges[1:]) / 2
    number_of_summary_fields = 2 + len(thresholds) + 3
    output_shape = (len(columns), number_of_bins + number_of_summary_fields)

    selected_source = source[columns]
    partition_summaries = []

    for partition in selected_source.to_delayed():
        summary_delayed = delayed(_qa_partition_distribution_summary)(
            partition,
            columns,
            value_range,
            histogram_edges,
            thresholds,
        )

        summary_array = da.from_delayed(
            summary_delayed,
            shape=output_shape,
            dtype="float64",
        )

        partition_summaries.append(summary_array)

    if not partition_summaries:
        return pd.DataFrame(), effective_bin_width

    total_summary = da.stack(partition_summaries, axis=0).sum(
        axis=0,
        dtype="float64",
        split_every=split_every,
    )

    # Histogram quantiles reuse the partition summaries and avoid a second scan.
    total_summary_result = total_summary.compute()

    histogram_counts = total_summary_result[:, :number_of_bins]
    if quantile_method == "histogram":
        quantile_result = pd.DataFrame(
            qa_histogram_quantiles(histogram_counts, histogram_edges, quantiles).T,
            index=quantiles,
            columns=columns,
        )
    else:
        distributed_quantiles = []
        for column in columns:
            values = selected_source[column]
            filtered = values.where((values >= lower_limit) & (values <= upper_limit))
            distributed_quantiles.append(filtered.quantile(q=list(quantiles)))
        quantile_results = dask.compute(*distributed_quantiles)
        quantile_result = pd.DataFrame(
            {column: result for column, result in zip(columns, quantile_results)}
        )
    field_index = number_of_bins

    counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    sums = total_summary_result[:, field_index]
    field_index += 1

    threshold_counts = {}

    for threshold in thresholds:
        threshold_counts[threshold] = total_summary_result[:, field_index].astype("int64")
        field_index += 1

    nonfinite_counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    below_range_counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    above_range_counts = total_summary_result[:, field_index].astype("int64")

    means = np.divide(
        sums,
        counts,
        out=np.full(len(columns), np.nan, dtype="float64"),
        where=counts > 0,
    )

    peak_bin_indices = np.argmax(histogram_counts, axis=1)
    peak_bin_centers = histogram_centers[peak_bin_indices].astype("float64")
    peak_bin_centers[counts == 0] = np.nan

    quantile_names = {
        0.16: "P16",
        0.50: "Median",
        0.84: "P84",
        0.95: "P95",
    }

    statistics_values = {
        "Column": columns,
        "Count": counts,
        "Mean": means,
    }

    for quantile in quantiles:
        column_name = quantile_names.get(quantile, f"P{quantile * 100:g}")
        statistics_values[column_name] = [
            quantile_result.loc[quantile, column]
            for column in columns
        ]

    statistics_values["Peak-bin center"] = peak_bin_centers

    for threshold in thresholds:
        fraction_name = f"Fraction ≤ {threshold:g}"
        statistics_values[fraction_name] = np.divide(
            threshold_counts[threshold],
            counts,
            out=np.full(len(columns), np.nan, dtype="float64"),
            where=counts > 0,
        )

    statistics_values.update(
        {
            "Non-finite": nonfinite_counts,
            "Below range": below_range_counts,
            "Above range": above_range_counts,
        }
    )

    statistics = pd.DataFrame(statistics_values)

    return statistics, effective_bin_width


# Plotting and display helpers

def ra_to_mollweide_x(ra_deg):
    """Convert RA in degrees to Mollweide longitude."""

    ra_centered = ((np.asarray(ra_deg) + 180.0) % 360.0) - 180.0
    return -np.deg2rad(ra_centered)


def plot_histogram_peak_marker(ax, counts, edges, color=None):
    """Draw a vertical marker at the most populated histogram bin."""

    counts = np.asarray(counts)

    if counts.size == 0 or not np.any(counts > 0):
        return

    peak_bin_index = int(np.argmax(counts))
    peak_bin_center = float((edges[peak_bin_index] + edges[peak_bin_index + 1]) / 2)
    peak_count = float(counts[peak_bin_index])

    _, peak_axis_y = ax.transAxes.inverted().transform(
        ax.transData.transform((peak_bin_center, peak_count))
    )
    peak_axis_y = float(np.clip(peak_axis_y, 0.0, 1.0))

    ax.plot(
        [peak_bin_center, peak_bin_center],
        [0.0, peak_axis_y],
        color=color,
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
        transform=ax.get_xaxis_transform(),
    )


def plot_wrapped_curve(ax, ra_deg, dec_deg, **plot_kwargs):
    """Plot a curve in Mollweide coordinates, split at RA wrap jumps."""

    x = ra_to_mollweide_x(ra_deg)
    y = np.deg2rad(dec_deg)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return

    jump_idx = np.where(np.abs(np.diff(x)) > np.pi)[0]
    start = 0
    first_segment = True

    for jump in jump_idx:
        end = jump + 1

        if end - start > 1:
            kwargs = plot_kwargs

            if not first_segment:
                kwargs = {
                    key: value
                    for key, value in plot_kwargs.items()
                    if key != "label"
                }

            ax.plot(x[start:end], y[start:end], **kwargs)
            first_segment = False

        start = end

    if len(x) - start > 1:
        kwargs = plot_kwargs

        if not first_segment:
            kwargs = {
                key: value
                for key, value in plot_kwargs.items()
                if key != "label"
            }

        ax.plot(x[start:], y[start:], **kwargs)


def add_band_model_metadata(
    statistics,
    bands,
    models,
    model_labels,
):
    """Add readable band and model columns to a QA table."""

    column_metadata = pd.DataFrame(
        [
            {
                "Column": f"{band}_{model}",
                "Band": band,
                "Model": model_labels.get(model, model),
            }
            for band in bands
            for model in models
        ]
    )

    return column_metadata.merge(
        statistics,
        on="Column",
        how="left",
    ).drop(columns="Column")


def style_distribution_statistics(
    statistics,
    caption,
    quantile_bin_width=None,
):
    """Apply consistent formatting to a distribution statistics table."""

    formatters = {
        "Count": "{:,}",
        "Mean": "{:.4f}",
        "Median": "{:.4f}",
        "P16": "{:.4f}",
        "P84": "{:.4f}",
        "P95": "{:.4f}",
        "Peak-bin center": "{:.4f}",
        "Non-finite": "{:,}",
        "Below range": "{:,}",
        "Above range": "{:,}",
    }

    for column in statistics.columns:
        if column.startswith("Fraction ≤"):
            formatters[column] = "{:.2%}"

    if quantile_bin_width is not None:
        decimals = max(0, int(np.ceil(-np.log10(quantile_bin_width) - 1e-10)))
        formatters["Peak-bin center"] = f"{{:.{decimals + 1}f}}"
    else:
        decimals = 4

    approximate_format = f"≈{{:.{decimals}f}}"
    for column in statistics.columns:
        if column == "Median" or (column.startswith("P") and column[1:2].isdigit()):
            formatters[column] = approximate_format

    # Use only formatters corresponding to columns actually present.
    formatters = {
        column: formatter
        for column, formatter in formatters.items()
        if column in statistics.columns
    }

    return (
        statistics.style
        .format(formatters)
        .hide(axis="index")
        .set_caption(caption)
    )


## Distributed execution


Create the configured Dask cluster for lazy QA operations.


In [ ]:
client, cluster = make_dask_cluster(cluster_config)

print(client)
print(cluster)


In [ ]:
def read_properties_file(path):
    """Read Java-style key=value metadata files used by HATS."""

    properties = {}

    with path.open(encoding="utf-8") as properties_file:
        for line in properties_file:
            stripped = line.strip()

            if not stripped or stripped.startswith("#") or "=" not in stripped:
                continue

            key, value = stripped.split("=", 1)
            properties[key.strip()] = value.strip()

    return properties


def is_hats_catalog_path(path_to_catalog):
    """Return True when a path points to a HATS catalog or collection."""

    if path_to_catalog.is_file():
        return False

    return (path_to_catalog / "collection.properties").is_file() or (
        path_to_catalog / "hats.properties"
    ).is_file()


def resolve_hats_data_path(path_to_catalog):
    """Return the directory containing the primary HATS catalog parquet files."""

    collection_properties_path = path_to_catalog / "collection.properties"

    if collection_properties_path.is_file():
        collection_properties = read_properties_file(collection_properties_path)
        primary_table_url = collection_properties.get("hats_primary_table_url")

        if not primary_table_url:
            raise ValueError(
                f"HATS collection is missing hats_primary_table_url: {collection_properties_path}"
            )

        return path_to_catalog / primary_table_url / "dataset"

    return path_to_catalog / "dataset"


def resolve_hats_properties_path(path_to_catalog):
    """Return the primary HATS catalog properties path."""

    collection_properties_path = path_to_catalog / "collection.properties"

    if collection_properties_path.is_file():
        collection_properties = read_properties_file(collection_properties_path)
        primary_table_url = collection_properties.get("hats_primary_table_url")

        if not primary_table_url:
            raise ValueError(
                f"HATS collection is missing hats_primary_table_url: {collection_properties_path}"
            )

        return path_to_catalog / primary_table_url / "hats.properties"

    return path_to_catalog / "hats.properties"


def resolve_catalog_files(catalog_config):
    """Resolve one non-HATS catalog path and return matching parquet files."""

    path_to_catalog = resolve_config_path(catalog_config["path"])
    parquet_pattern = catalog_config.get("parquet_pattern", "*.parquet")

    if path_to_catalog.is_file():
        parquet_files = [path_to_catalog]
    else:
        parquet_files = sorted(path_to_catalog.rglob(parquet_pattern))

    if not parquet_files:
        raise FileNotFoundError(
            f"No parquet files found for catalog path: {path_to_catalog}"
        )

    return path_to_catalog, parquet_files


def make_catalog_context(catalog_config):
    """Open a configured catalog and return the data needed by QA sections."""

    path_to_catalog = resolve_config_path(catalog_config["path"])

    if is_hats_catalog_path(path_to_catalog):
        try:
            import lsdb
        except ImportError as error:
            raise ImportError(
                "lsdb is required for HATS catalog inputs. Install it with: "
                "conda install -c conda-forge lsdb"
            ) from error

        primary_catalog_path = resolve_hats_properties_path(path_to_catalog).parent
        # QA does not need the collection's margin cache. Opening the primary
        # table directly avoids loading and transforming its margin partitions.
        lsdb_catalog = lsdb.open_catalog(primary_catalog_path, columns="all")
        parquet_pattern = catalog_config.get("parquet_pattern", "*.parquet")
        data_path = resolve_hats_data_path(path_to_catalog)
        hats_properties = read_properties_file(primary_catalog_path / "hats.properties")
        parquet_files = sorted(data_path.rglob(parquet_pattern))

        if not parquet_files:
            raise FileNotFoundError(
                f"No HATS parquet files found below primary catalog data path: {data_path}"
            )

        return {
            "kind": "hats",
            "path": path_to_catalog,
            "primary_catalog_path": primary_catalog_path,
            "parquet_files": parquet_files,
            "dataframe": lsdb_catalog,
            "lsdb_catalog": lsdb_catalog,
            "hats_properties": hats_properties,
        }

    _, parquet_files = resolve_catalog_files(catalog_config)

    return {
        "kind": "parquet",
        "path": path_to_catalog,
        "parquet_files": parquet_files,
        "dataframe": dd.read_parquet(parquet_files, engine="pyarrow"),
        "lsdb_catalog": None,
        "hats_properties": {},
    }


def read_catalog_columns(catalog_context, columns=None):
    """Read selected columns from a parquet or HATS-backed catalog context."""

    if catalog_context["kind"] == "hats":
        if columns is None:
            return catalog_context["lsdb_catalog"]

        import lsdb

        # Selecting columns from an already opened Catalog happens after the
        # Parquet read. Pass them to open_catalog for actual read projection.
        selected_catalog = lsdb.open_catalog(
            catalog_context["primary_catalog_path"], columns=list(columns)
        )
        return selected_catalog[list(columns)]

    return dd.read_parquet(
        catalog_context["parquet_files"],
        engine="pyarrow",
        columns=columns,
    )


def read_catalog_healpix_column(catalog_context, healpix_column):
    """Read a HEALPix column, including HATS internal columns hidden by LSDB."""

    if healpix_column in catalog_context["dataframe"].columns:
        return read_catalog_columns(catalog_context, columns=[healpix_column])

    return dd.read_parquet(
        catalog_context["parquet_files"],
        engine="pyarrow",
        columns=[healpix_column],
    )


def format_bytes(size_bytes):
    """Format a byte count with binary units."""

    units = ("B", "KiB", "MiB", "GiB", "TiB", "PiB")
    value = float(size_bytes)

    for unit in units:
        if value < 1024.0 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "B" else f"{int(value)} {unit}"
        value /= 1024.0


def catalog_size_bytes(catalog_context):
    """Compute catalog size from selected parquet files."""

    if catalog_context["path"].is_file():
        return catalog_context["path"].stat().st_size

    return sum(path.stat().st_size for path in catalog_context["parquet_files"])


def display_section_heading(title):
    """Display a third-level report section heading."""

    display(Markdown(f"### {title}"))


def display_catalog_columns(columns):
    """Display catalog column names in a scrollable text area."""

    cols_text = "\n".join(map(str, columns))
    display(
        HTML(
            f"""
            <textarea
                rows="10"
                style="width: 100%; font-family: monospace; white-space: pre;"
                readonly
            >{html.escape(cols_text)}</textarea>
            """
        )
    )


def display_unique_values(column, values, rows=10):
    """Display unique values for one column in a scrollable text area."""

    values_text = "\n".join(map(str, values))
    display(
        HTML(
            f"""
            <p style="margin: 8px 0 4px 0;"><b>Unique values for {html.escape(str(column))}</b></p>
            <textarea
                rows="{int(rows)}"
                style="width: 100%; font-family: monospace; white-space: pre;"
                readonly
            >{html.escape(values_text)}</textarea>
            """
        )
    )


def render_catalog_basics(catalog_config):
    """Render required size, row-count, and column-count information."""

    display_section_heading("Basic Product Information")
    qa_log(f"[{catalog_config['title']}] Resolving catalog input files")
    catalog_context = make_catalog_context(catalog_config)
    catalog = catalog_context["dataframe"]
    qa_log(f"[{catalog_config['title']}] Computing catalog size")

    if not catalog_config.get("omit_paths", False):
        print("Catalog path:", catalog_context["path"])
    print("Catalog type:", "HATS" if catalog_context["kind"] == "hats" else "Parquet")
    print("Parquet files:", f"{len(catalog_context['parquet_files']):,}")
    print("Catalog size:", format_bytes(catalog_size_bytes(catalog_context)))

    qa_log(f"[{catalog_config['title']}] Computing total row count")
    if catalog_context["kind"] == "hats":
        # HATS stores the row count in catalog metadata.
        qa_total_rows = len(catalog)
    else:
        count_column = catalog.columns[0]
        count_data = read_catalog_columns(catalog_context, columns=[count_column])
        qa_total_rows = qa_row_count(count_data)
        del count_data

    print("Total rows:", f"{qa_total_rows:,}")
    catalog_context["total_rows"] = qa_total_rows
    qa_log(f"[{catalog_config['title']}] Computing number of columns")
    print("Total columns:", f"{len(catalog.columns):,}")
    qa_log(f"[{catalog_config['title']}] Rendering column names")
    display_catalog_columns(catalog.columns)

    return catalog_context


def render_unique_count(catalog_config, catalog_context):
    """Render exact unique-count diagnostics when configured."""

    unique_count_config = catalog_config.get("unique_count")

    if unique_count_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing unique count")
    display_section_heading("Unique Count")
    unique_column = unique_count_config["column"]
    max_unique_values = int(unique_count_config.get("max_unique_values", 10000))
    list_values = bool(unique_count_config.get("list_values", False))
    list_rows = int(unique_count_config.get("list_rows", 10))

    print(f"Computing exact global unique count for column '{unique_column}'.")
    print(f"Driver collection cap: {max_unique_values:,} unique values.")

    unique_data = read_catalog_columns(catalog_context, columns=[unique_column])
    qa_unique_values = qa_unique_count(
        unique_data,
        unique_column,
        max_unique_values=max_unique_values,
        return_values=list_values,
    )
    unique_values = None

    if list_values:
        qa_unique_values, unique_values = qa_unique_values

    print(f"Exact global unique count for column '{unique_column}': {qa_unique_values:,}")

    if unique_values is not None:
        display_unique_values(unique_column, unique_values, rows=list_rows)

    del unique_data


def get_survey_area_config(catalog_config):
    """Return normalized survey-area configuration when explicitly enabled."""

    if "survey_area" not in catalog_config or catalog_config["survey_area"] is False:
        return None

    survey_area_config = catalog_config["survey_area"]

    if survey_area_config in (None, True):
        survey_area_config = {}

    if not isinstance(survey_area_config, dict):
        raise ValueError(
            f"survey_area for catalog '{catalog_config['title']}' must be "
            "a YAML mapping, true, false, or null."
        )

    return dict(survey_area_config)


def resolve_survey_area_coordinate_columns(catalog_config, survey_area_config):
    """Resolve coordinate columns used for survey-area estimation."""

    spatial_distribution_config = catalog_config.get("spatial_distribution") or {}
    ra_column = survey_area_config.get(
        "ra_column",
        spatial_distribution_config.get("ra_column", "coord_ra"),
    )
    dec_column = survey_area_config.get(
        "dec_column",
        spatial_distribution_config.get("dec_column", "coord_dec"),
    )

    return ra_column, dec_column


def resolve_survey_area_healpix_column(catalog_context, survey_area_config, target_order):
    """Resolve an existing HEALPix column suitable for area estimation."""

    if not survey_area_config.get("use_hats_healpix_column", True):
        return None, None

    hats_properties = catalog_context.get("hats_properties", {})
    healpix_column = survey_area_config.get(
        "healpix_column",
        hats_properties.get("hats_col_healpix"),
    )
    healpix_order = survey_area_config.get(
        "healpix_column_order",
        hats_properties.get("hats_col_healpix_order"),
    )

    if healpix_column is None or healpix_order is None:
        return None, None

    if healpix_column not in catalog_context["dataframe"].columns and catalog_context["kind"] != "hats":
        if "healpix_column" in survey_area_config:
            raise ValueError(
                f"Configured survey_area.healpix_column is not present in the catalog: {healpix_column}"
            )

        return None, None

    healpix_order = int(healpix_order)

    if healpix_order < int(target_order):
        if "healpix_column" in survey_area_config:
            raise ValueError(
                "Configured survey_area.healpix_column_order must be greater than "
                "or equal to survey_area.order."
            )

        return None, None

    return healpix_column, healpix_order


def render_survey_area(catalog_config, catalog_context):
    """Render survey area and mean object density when configured."""

    survey_area_config = get_survey_area_config(catalog_config)

    if survey_area_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing survey area")
    display_section_heading("Survey Area")
    ra_column, dec_column = resolve_survey_area_coordinate_columns(
        catalog_config,
        survey_area_config,
    )
    order = int(survey_area_config.get("order", 12))
    split_every = int(survey_area_config.get("split_every", 8))
    split_out = int(survey_area_config.get("split_out", 64))
    shuffle_method = survey_area_config.get("shuffle_method")
    healpix_column, healpix_order = resolve_survey_area_healpix_column(
        catalog_context,
        survey_area_config,
        order,
    )

    if healpix_column:
        print(
            "Computing exact occupied HEALPix pixels by degrading "
            f"'{healpix_column}' from order {healpix_order} to order {order}."
        )
    else:
        print(
            "Computing exact occupied HEALPix pixels from "
            f"'{ra_column}' and '{dec_column}' at order {order}."
        )
    print(f"Dask global deduplication split_out: {split_out:,} partitions.")
    print(
        "Survey area is a HEALPix-grid estimate: each occupied pixel at the "
        "reported order contributes its full area, including boundary pixels."
    )

    if healpix_column:
        healpix_data = read_catalog_healpix_column(catalog_context, healpix_column)
        area_result = qa_survey_area_from_healpix(
            healpix_data,
            healpix_column,
            healpix_order,
            target_order=order,
            split_every=split_every,
            split_out=split_out,
            shuffle_method=shuffle_method,
        )
        del healpix_data
        pixel_source = f"{healpix_column} (order {healpix_order})"
    else:
        coordinate_data = read_catalog_columns(
            catalog_context,
            columns=[ra_column, dec_column],
        )
        area_result = qa_survey_area(
            coordinate_data,
            ra_column,
            dec_column,
            order=order,
            split_every=split_every,
            split_out=split_out,
            shuffle_method=shuffle_method,
        )
        del coordinate_data
        pixel_source = f"{ra_column}, {dec_column}"

    total_rows = catalog_context["total_rows"]
    area_sq_deg = area_result["area_sq_deg"]
    density = total_rows / area_sq_deg if area_sq_deg > 0 else np.nan
    summary = pd.DataFrame(
        [
            {
                "RA column": ra_column,
                "Dec column": dec_column,
                "Pixel source": pixel_source,
                "HEALPix order": area_result["order"],
                "NSIDE": area_result["nside"],
                "Occupied pixels": area_result["unique_pixels"],
                "Pixel area (deg²)": area_result["pixel_area_sq_deg"],
                "Survey area (deg²)": area_sq_deg,
                "Total rows": total_rows,
                "Mean density (objects/deg²)": density,
            }
        ]
    )
    display(
        summary.style.format(
            {
                "Occupied pixels": "{:,}",
                "Pixel area (deg²)": "{:.8f}",
                "Survey area (deg²)": "{:,.4f}",
                "Total rows": "{:,}",
                "Mean density (objects/deg²)": "{:,.4f}",
            }
        ).hide(axis="index")
    )


def render_basic_statistics(catalog_config, catalog_context):
    """Render configured column statistics from data or HATS metadata."""

    basic_statistics_config = catalog_config.get("basic_statistics")

    if basic_statistics_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing basic statistics")
    display_section_heading("Basic Statistics for Selected Columns")
    statistics_columns = get_basic_statistics_columns(
        catalog_context["dataframe"].columns,
        basic_statistics_config,
    )

    if catalog_context["kind"] == "hats":
        qa_log(f"[{catalog_config['title']}] Reading HATS column metadata")
        statistics = catalog_context["lsdb_catalog"].aggregate_column_statistics(
            include_columns=statistics_columns,
            use_default_columns=False,
        ).reindex(statistics_columns)
        statistics.index.name = "Column"
        metadata_rows = pd.to_numeric(statistics["row_count"], errors="coerce")
        incomplete = metadata_rows.ne(catalog_context["total_rows"])
        if incomplete.any():
            missing_columns = statistics.index[incomplete].tolist()
            print(
                "Complete footer statistics are unavailable for: "
                + ", ".join(missing_columns)
                + ". Their statistics are left blank."
            )
            statistics.loc[incomplete, :] = None
        statistics = statistics.rename(
            columns={
                "row_count": "Rows",
                "null_count": "Nulls",
                "min_value": "Minimum",
                "max_value": "Maximum",
            }
        )
        statistics = statistics.reindex(columns=["Rows", "Nulls", "Minimum", "Maximum"])
        print(
            "HATS statistics come from Parquet metadata and do not scan catalog rows. "
            "Rows includes null entries; Nulls counts them separately. "
            "Mean, standard deviation, and percentiles are not available in this summary."
        )
    else:
        statistics_data = read_catalog_columns(
            catalog_context,
            columns=statistics_columns,
        )
        statistics = statistics_data.describe().compute()
        del statistics_data
        print(
            "Dask describe() count excludes null entries, and its percentiles are approximate."
        )
        omitted_columns = [
            column for column in statistics_columns if column not in statistics.columns
        ]
        if omitted_columns:
            print(
                "Dask describe() omitted these selected columns under its default "
                "data-type selection: " + ", ".join(omitted_columns) + "."
            )

    statistics_html = (
        '<div style="max-height: 520px; overflow: auto;">'
        + statistics.to_html(max_rows=None, max_cols=None)
        + "</div>"
    )

    display(HTML(statistics_html))


def render_spatial_distribution(catalog_config, catalog_context):
    """Render the spatial distribution plot when configured."""

    spatial_distribution_config = catalog_config.get("spatial_distribution")

    if spatial_distribution_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating spatial distribution plot")
    display_section_heading("Spatial Distribution")
    plot_data = read_catalog_columns(
        catalog_context,
        columns=[
            spatial_distribution_config["ra_column"],
            spatial_distribution_config["dec_column"],
        ],
    )

    xbins = np.linspace(-np.pi, np.pi, int(spatial_distribution_config["ra_edge_count"]))
    ybins = np.linspace(
        -np.pi / 2.0,
        np.pi / 2.0,
        int(spatial_distribution_config["dec_edge_count"]),
    )

    H = qa_histogram2d(
        plot_data,
        spatial_distribution_config["ra_column"],
        spatial_distribution_config["dec_column"],
        xbins,
        ybins,
        split_every=int(spatial_distribution_config.get("split_every", 8)),
    )

    H = np.ma.masked_equal(H, 0)
    fig = plt.figure(figsize=(16, 8))
    ax = fig.add_subplot(111, projection="mollweide")

    if H.count() > 0:
        mesh = ax.pcolormesh(
            xbins,
            ybins,
            H.T,
            norm=LogNorm(),
            shading="auto",
            cmap="viridis",
            zorder=1,
        )
        cbar = fig.colorbar(mesh, ax=ax, pad=0.05)
        cbar.set_label("Number of objects")
    else:
        ax.text(
            0.5,
            0.5,
            "No finite coordinate pairs",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )

    ax.grid(False)

    dec_grid = np.deg2rad(np.linspace(-90, 90, 500))
    for grid_ra_deg in np.arange(-150, 181, 30):
        ax.plot(
            np.full_like(dec_grid, np.deg2rad(grid_ra_deg)),
            dec_grid,
            color="gray",
            linewidth=0.6,
            alpha=0.5,
            zorder=2,
        )

    ra_grid = np.deg2rad(np.linspace(-180, 180, 800))
    for grid_dec_deg in np.arange(-75, 76, 15):
        ax.plot(
            ra_grid,
            np.full_like(ra_grid, np.deg2rad(grid_dec_deg)),
            color="gray",
            linewidth=0.6,
            alpha=0.5,
            zorder=2,
        )

    for footprint_config in spatial_distribution_config.get("footprints", []):
        footprint = pd.read_csv(resolve_config_path(footprint_config["path"]))
        footprint_type = footprint_config.get("type", "curve")

        if footprint_type == "regions":
            first_curve = True
            region_column = footprint_config.get("region_column", "region_id")
            ring_type_column = footprint_config.get("ring_type_column", "ring_type")
            exterior_value = footprint_config.get("exterior_value", "exterior")
            vertex_column = footprint_config.get("vertex_column", "vertex_id")

            for _, footprint_region in footprint.groupby(region_column, sort=False):
                exterior = (
                    footprint_region[footprint_region[ring_type_column] == exterior_value]
                    .sort_values(vertex_column)
                )

                plot_wrapped_curve(
                    ax,
                    exterior[footprint_config["ra_column"]].to_numpy(),
                    exterior[footprint_config["dec_column"]].to_numpy(),
                    linewidth=footprint_config.get("linewidth", 1),
                    color=footprint_config.get("color"),
                    zorder=footprint_config.get("zorder", 3),
                    label=footprint_config["label"] if first_curve else "_nolegend_",
                )
                first_curve = False

        elif footprint_type == "curve":
            sort_by = footprint_config.get("sort_by")
            if sort_by:
                footprint = footprint.sort_values(sort_by)

            plot_wrapped_curve(
                ax,
                footprint[footprint_config["ra_column"]].to_numpy(),
                footprint[footprint_config["dec_column"]].to_numpy(),
                linewidth=footprint_config.get("linewidth", 1),
                color=footprint_config.get("color"),
                zorder=footprint_config.get("zorder", 3),
                label=footprint_config["label"],
            )
        else:
            raise ValueError(f"Unsupported footprint type: {footprint_type}")

    tick_degs = np.array([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150])
    tick_labels = [
        "150 deg",
        "120 deg",
        "90 deg",
        "60 deg",
        "30 deg",
        "0 deg",
        "330 deg",
        "300 deg",
        "270 deg",
        "240 deg",
        "210 deg",
    ]

    ax.set_xticks(np.deg2rad(tick_degs))
    ax.set_xticklabels(tick_labels)
    ax.set_xlabel(spatial_distribution_config["ra_column"])
    ax.set_ylabel(spatial_distribution_config["dec_column"])
    ax.set_title(
        f"{catalog_config['title']} - {spatial_distribution_config['title_suffix']}"
    )

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
    del plot_data
    del H


def render_magnitude_histograms(catalog_config, catalog_context):
    """Render magnitude histograms when configured."""

    magnitudes_config = catalog_config.get("magnitudes")

    if magnitudes_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating magnitude histograms")
    display_section_heading("Magnitude Histogram")
    bands = magnitudes_config["bands"]
    magnitude_models = magnitudes_config["models"]
    model_labels = magnitudes_config["model_labels"]
    model_colors = magnitudes_config["model_colors"]
    magnitude_histogram_config = magnitudes_config["histogram"]

    plot_data, magnitude_columns = make_flux_magnitude_source(
        catalog_context,
        bands=bands,
        models=magnitude_models,
        section_config=magnitudes_config,
    )

    magnitude_histograms, magnitude_edges = qa_histograms1d(
        plot_data,
        columns=magnitude_columns,
        bins=int(magnitude_histogram_config["bins"]),
        value_range=tuple(magnitude_histogram_config["range"]),
        split_every=int(magnitude_histogram_config.get("split_every", 8)),
    )

    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 15), sharex=True)
    axes = axes.ravel()

    for ax, band in zip(axes, bands):
        peak_marker_specs = []

        for model in magnitude_models:
            column = f"{band}_{model}"
            color = model_colors.get(model)
            ax.stairs(
                values=magnitude_histograms[column],
                edges=magnitude_edges,
                label=model_labels.get(model, model),
                color=color,
                linewidth=1.8,
            )
            peak_marker_specs.append((magnitude_histograms[column], color))

        ax.set_title(f"{band}-band magnitude")
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Count")
        ax.set_xlim(magnitude_histogram_config["range"][0], magnitude_histogram_config["range"][1])
        ax.set_yscale("log")

        for counts, color in peak_marker_specs:
            plot_histogram_peak_marker(
                ax,
                counts=counts,
                edges=magnitude_edges,
                color=color,
            )

        ax.grid(alpha=0.2)
        ax.legend(title="Model")
        ax.tick_params(axis="x", which="both", labelbottom=True)

    fig.suptitle("Magnitude Distributions by Model", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
    del plot_data
    del magnitude_histograms
    del magnitude_edges


def render_magnitude_statistics(catalog_config, catalog_context):
    """Render magnitude statistics when configured."""

    magnitudes_config = catalog_config.get("magnitudes")

    if magnitudes_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing magnitude statistics")
    display_section_heading("Magnitude Statistics")
    bands = magnitudes_config["bands"]
    magnitude_models = magnitudes_config["models"]
    model_labels = magnitudes_config["model_labels"]
    magnitude_statistics_config = magnitudes_config["statistics"]

    statistics_data, magnitude_columns = make_flux_magnitude_source(
        catalog_context,
        bands=bands,
        models=magnitude_models,
        section_config=magnitudes_config,
    )

    statistics_range = tuple(magnitude_statistics_config["range"])
    peak_bin_width = float(magnitude_statistics_config["peak_bin_width"])
    quantile_method = magnitude_statistics_config.get(
        "quantile_method",
        "histogram" if catalog_context["kind"] == "hats" else "dask",
    )
    magnitude_statistics, effective_bin_width = qa_distribution_statistics(
        statistics_data,
        columns=magnitude_columns,
        value_range=statistics_range,
        peak_bin_width=peak_bin_width,
        thresholds=tuple(magnitude_statistics_config.get("thresholds", [])),
        quantiles=tuple(magnitude_statistics_config.get("quantiles", [0.16, 0.50, 0.84, 0.95])),
        split_every=int(magnitude_statistics_config.get("split_every", 8)),
        quantile_method=quantile_method,
    )

    magnitude_statistics = add_band_model_metadata(
        magnitude_statistics,
        bands=bands,
        models=magnitude_models,
        model_labels=model_labels,
    )

    print(
        "Statistics include only finite magnitude values in the range "
        f"{statistics_range[0]} ≤ magnitude ≤ {statistics_range[1]}."
    )

    if any(model_uses_flux_conversion(model) for model in magnitude_models):
        print(
            "Flux columns were converted to magnitudes with "
            f"magnitude = {float(magnitudes_config['mag_offset']):g} - 2.5 log10(flux). "
            "Non-positive, missing, and non-finite flux values are converted to NaN "
            "and excluded from finite-value histograms and statistics."
        )

    if quantile_method == "histogram":
        print(
            "Mean is calculated from filtered values. Median and other quantiles "
            f"are estimated from {effective_bin_width:.3g}-mag bins; values within "
            "each bin are interpolated."
        )
    else:
        print(
            "Mean is calculated from filtered values. Median and other quantiles "
            "are approximate results of Dask's distributed quantile algorithm."
        )
    print(
        f"Peak-bin center is the center of the most populated {effective_bin_width:.3g}-mag bin "
        "and should be interpreted as an empirical distribution turnover, not as a calibrated completeness limit."
    )
    print(
        "Non-finite, below-range, and above-range values are reported separately and excluded from the descriptive statistics."
    )

    display(
        style_distribution_statistics(
            magnitude_statistics,
            caption=(
                "Magnitude statistics by band and measurement model"
                + (
                    f" (≈ quantiles from {effective_bin_width:.3g}-mag bins)"
                    if quantile_method == "histogram"
                    else " (≈ quantiles from Dask)"
                )
            ),
            quantile_bin_width=(
                effective_bin_width if quantile_method == "histogram" else None
            ),
        )
    )
    del statistics_data
    del magnitude_statistics



def render_magnitude_error_trends(catalog_config, catalog_context):
    """Render binned magnitude versus magnitude-error trends when configured."""

    trend_config = catalog_config.get("magnitude_error_trends")

    if trend_config is None:
        return

    magnitudes_config = catalog_config.get("magnitudes")
    magnitude_errors_config = catalog_config.get("magnitude_errors")

    if magnitudes_config is None or magnitude_errors_config is None:
        raise ValueError(
            "magnitude_error_trends requires both magnitudes and magnitude_errors."
        )

    qa_log(f"[{catalog_config['title']}] Generating magnitude-error trend plots")
    display_section_heading("Magnitude Error Trends")

    bands = trend_config.get("bands")

    if bands is None:
        band = trend_config.get("band")
        bands = [band] if band is not None else magnitudes_config["bands"]

    magnitude_models, error_models = get_magnitude_error_trend_models(
        trend_config,
        magnitudes_config,
        magnitude_errors_config,
    )
    model_labels = magnitudes_config["model_labels"]
    model_colors = magnitudes_config["model_colors"]

    if catalog_context["kind"] == "hats":
        # Build both derived sets on one Catalog so partition alignment is retained.
        plot_data = make_magnitude_error_trend_source(
            catalog_context,
            bands,
            magnitude_models,
            error_models,
            magnitudes_config,
        )
    else:
        magnitude_data, _ = make_flux_magnitude_source(
            catalog_context,
            bands=bands,
            models=magnitude_models,
            section_config=magnitudes_config,
        )
        error_data, _ = make_magnitude_error_source(
            catalog_context,
            bands=bands,
            models=error_models,
            section_config=magnitude_errors_config,
        )
        plot_data = dd.concat([magnitude_data, error_data], axis=1)
    magnitude_columns = make_band_model_columns(bands, magnitude_models)
    error_columns = make_band_model_columns(bands, error_models)
    column_pairs = list(zip(magnitude_columns, error_columns))
    magnitude_range = tuple(trend_config["magnitude_range"])
    error_range = tuple(trend_config["error_range"])
    dispersion_quantiles = tuple(
        trend_config.get("dispersion_quantiles", [0.16, 0.84])
    )

    relation_histograms, magnitude_edges, error_edges = qa_binned_relation_histograms(
        plot_data,
        column_pairs=column_pairs,
        x_bins=int(trend_config.get("magnitude_bins", trend_config.get("bins", 50))),
        y_bins=int(trend_config.get("error_bins", trend_config.get("bins", 50))),
        x_range=magnitude_range,
        y_range=error_range,
        split_every=int(trend_config.get("split_every", 8)),
    )

    print(
        "Trend lines use error-bin centers to estimate mean error; shaded "
        "quantiles are also estimated from the 2D histogram. "
        f"Bin widths: {magnitude_edges[1] - magnitude_edges[0]:.3g} mag and "
        f"{error_edges[1] - error_edges[0]:.3g} mag error."
    )

    magnitude_centers = (magnitude_edges[:-1] + magnitude_edges[1:]) / 2
    min_count = int(trend_config.get("min_count", 1))
    fill_alpha = float(trend_config.get("fill_alpha", 0.15))
    linewidth = float(trend_config.get("linewidth", 2.0))

    ncols = int(trend_config.get("ncols", 2))
    nrows = int(np.ceil(len(bands) / ncols))
    default_figsize = [7 * ncols, 5 * nrows]
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=tuple(trend_config.get("figsize", default_figsize)),
        sharex=True,
        sharey=True,
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, band in zip(axes, bands):
        for magnitude_model, error_model in zip(magnitude_models, error_models):
            column_pair = (f"{band}_{magnitude_model}", f"{band}_{error_model}")
            counts, mean_error, quantile_values = summarize_binned_relation(
                relation_histograms[column_pair],
                error_edges,
                quantiles=dispersion_quantiles,
            )
            valid = counts >= min_count
            color = model_colors.get(magnitude_model)
            label = model_labels.get(magnitude_model, magnitude_model)

            if len(quantile_values) >= 2:
                ax.fill_between(
                    magnitude_centers,
                    quantile_values[0],
                    quantile_values[-1],
                    where=valid,
                    color=color,
                    alpha=fill_alpha,
                    linewidth=0,
                )

            ax.plot(
                magnitude_centers[valid],
                mean_error[valid],
                color=color,
                linewidth=linewidth,
                label=label,
            )

        ax.set_title(f"{band}-band magnitude error trend")
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Mean magnitude error")
        ax.set_xlim(magnitude_range[0], magnitude_range[1])
        ax.set_ylim(error_range[0], error_range[1])
        ax.grid(alpha=0.2)
        ax.legend(title="Model")
        ax.tick_params(axis="x", which="both", labelbottom=True)

    for ax in axes[len(bands):]:
        ax.set_visible(False)

    fig.suptitle("Magnitude Error Trends by Model", fontsize=16, y=1.01)

    plt.tight_layout()
    plt.show()
    del plot_data
    del relation_histograms
    del magnitude_edges
    del error_edges


def render_magnitude_error_histograms(catalog_config, catalog_context):
    """Render magnitude-error histograms when configured."""

    magnitude_errors_config = catalog_config.get("magnitude_errors")

    if magnitude_errors_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating magnitude-error histograms")
    display_section_heading("Magnitude Error Histogram")
    magnitudes_config = catalog_config.get("magnitudes")
    bands = magnitude_errors_config.get(
        "bands",
        magnitudes_config["bands"] if magnitudes_config else None,
    )

    if bands is None:
        raise ValueError(
            "magnitude_errors.bands is required when magnitudes is not configured."
        )

    magnitude_error_models = magnitude_errors_config["models"]
    model_labels = magnitude_errors_config["model_labels"]
    model_colors = magnitude_errors_config["model_colors"]
    magnitude_error_histogram_config = magnitude_errors_config["histogram"]

    plot_data, magnitude_error_columns = make_magnitude_error_source(
        catalog_context,
        bands=bands,
        models=magnitude_error_models,
        section_config=magnitude_errors_config,
    )

    magnitude_error_histograms, magnitude_error_edges = qa_histograms1d(
        plot_data,
        columns=magnitude_error_columns,
        bins=int(magnitude_error_histogram_config["bins"]),
        value_range=tuple(magnitude_error_histogram_config["range"]),
        split_every=int(magnitude_error_histogram_config.get("split_every", 8)),
    )

    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 15), sharex=True)
    axes = axes.ravel()

    for ax, band in zip(axes, bands):
        for model in magnitude_error_models:
            column = f"{band}_{model}"
            ax.stairs(
                values=magnitude_error_histograms[column],
                edges=magnitude_error_edges,
                label=model_labels.get(model, model),
                color=model_colors.get(model),
                linewidth=1.8,
            )

        ax.set_title(f"{band}-band magnitude error")
        ax.set_xlabel("Magnitude error")
        ax.set_ylabel("Count")
        ax.set_xlim(
            magnitude_error_histogram_config["range"][0],
            magnitude_error_histogram_config["range"][1],
        )
        ax.set_yscale("log")
        ax.grid(alpha=0.2)
        ax.legend(title="Model")
        ax.tick_params(axis="x", which="both", labelbottom=True)

    fig.suptitle("Magnitude Error Distributions by Model", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
    del plot_data
    del magnitude_error_histograms
    del magnitude_error_edges


def render_magnitude_error_statistics(catalog_config, catalog_context):
    """Render magnitude-error statistics when configured."""

    magnitude_errors_config = catalog_config.get("magnitude_errors")

    if magnitude_errors_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing magnitude-error statistics")
    display_section_heading("Magnitude Error Statistics")
    magnitudes_config = catalog_config.get("magnitudes")
    bands = magnitude_errors_config.get(
        "bands",
        magnitudes_config["bands"] if magnitudes_config else None,
    )

    if bands is None:
        raise ValueError(
            "magnitude_errors.bands is required when magnitudes is not configured."
        )

    magnitude_error_models = magnitude_errors_config["models"]
    model_labels = magnitude_errors_config["model_labels"]
    magnitude_error_statistics_config = magnitude_errors_config["statistics"]

    statistics_data, magnitude_error_columns = make_magnitude_error_source(
        catalog_context,
        bands=bands,
        models=magnitude_error_models,
        section_config=magnitude_errors_config,
    )

    statistics_range = tuple(magnitude_error_statistics_config["range"])
    peak_bin_width = float(magnitude_error_statistics_config["peak_bin_width"])
    error_thresholds = tuple(magnitude_error_statistics_config.get("thresholds", []))
    quantile_method = magnitude_error_statistics_config.get(
        "quantile_method",
        "histogram" if catalog_context["kind"] == "hats" else "dask",
    )

    magnitude_error_statistics, effective_bin_width = qa_distribution_statistics(
        statistics_data,
        columns=magnitude_error_columns,
        value_range=statistics_range,
        peak_bin_width=peak_bin_width,
        thresholds=error_thresholds,
        quantiles=tuple(magnitude_error_statistics_config.get("quantiles", [0.16, 0.50, 0.84, 0.95])),
        split_every=int(magnitude_error_statistics_config.get("split_every", 8)),
        quantile_method=quantile_method,
    )

    magnitude_error_statistics = add_band_model_metadata(
        magnitude_error_statistics,
        bands=bands,
        models=magnitude_error_models,
        model_labels=model_labels,
    )

    print(
        "Statistics include only finite magnitude-error values in the range "
        f"{statistics_range[0]} ≤ magnitude error ≤ {statistics_range[1]}."
    )

    if any(model_uses_flux_error_conversion(model) for model in magnitude_error_models):
        print(
            "Flux-error columns were converted to magnitude errors with "
            "sigma_mag = 2.5 / ln(10) * flux_err / flux. Non-positive, missing, "
            "and non-finite flux or flux-error values are converted to NaN and "
            "excluded from finite-value histograms and statistics."
        )

    if quantile_method == "histogram":
        print(
            "Mean is calculated from filtered values. Median and other quantiles "
            f"are estimated from {effective_bin_width:.3g}-mag-error bins; values "
            "within each bin are interpolated."
        )
    else:
        print(
            "Mean is calculated from filtered values. Median and other quantiles "
            "are approximate results of Dask's distributed quantile algorithm."
        )
    print(
        f"Peak-bin center is the center of the most populated {effective_bin_width:.3g}-mag bin."
    )
    print(
        "Threshold fractions use Count as their denominator. Non-finite, below-range, "
        "and above-range values are reported separately and excluded from the descriptive statistics."
    )

    display(
        style_distribution_statistics(
            magnitude_error_statistics,
            caption=(
                "Magnitude-error statistics by band and measurement model"
                + (
                    f" (≈ quantiles from {effective_bin_width:.3g}-mag-error bins)"
                    if quantile_method == "histogram"
                    else " (≈ quantiles from Dask)"
                )
            ),
            quantile_bin_width=(
                effective_bin_width if quantile_method == "histogram" else None
            ),
        )
    )
    del statistics_data
    del magnitude_error_statistics


def get_hats_plot_kwargs(catalog_config, section_name):
    """Return keyword arguments for an optional HATS plot section."""

    section_config = catalog_config.get(section_name)

    if section_config in (None, True):
        return {}

    if not isinstance(section_config, dict):
        raise ValueError(
            f"{section_name} for catalog '{catalog_config['title']}' must be "
            "a YAML mapping, true, false, or null."
        )

    return dict(section_config)


def get_configured_hats_plot_sections(catalog_config):
    """Return configured HATS-only plot section names."""

    return [
        section_name
        for section_name in ("plot_pixels", "plot_coverage")
        if section_name in catalog_config
    ]


def validate_hats_plot_sections(catalog_config, catalog_context):
    """Fail if HATS-only plot sections are configured for a non-HATS input."""

    configured_hats_plot_sections = get_configured_hats_plot_sections(catalog_config)

    if configured_hats_plot_sections and catalog_context["kind"] != "hats":
        raise ValueError(
            f"Catalog '{catalog_config['title']}' configures HATS-only plot "
            f"section(s): {', '.join(configured_hats_plot_sections)}, "
            "but its input path is not a HATS catalog or collection."
        )

    return configured_hats_plot_sections


def render_hats_catalog_plots(catalog_config, catalog_context):
    """Render LSDB-specific HATS catalog maps."""

    configured_hats_plot_sections = validate_hats_plot_sections(
        catalog_config,
        catalog_context,
    )

    if not configured_hats_plot_sections:
        return

    lsdb_catalog = catalog_context["lsdb_catalog"]

    if catalog_config.get("plot_pixels", False) is not False:
        qa_log(f"[{catalog_config['title']}] Generating HATS pixel-density map")
        display_section_heading("HATS Pixel Density")
        pixels_kwargs = get_hats_plot_kwargs(catalog_config, "plot_pixels")
        projection = pixels_kwargs.pop("projection", "MOL")
        fig, ax = lsdb_catalog.plot_pixels(projection=projection, **pixels_kwargs)
        ax.set_title(f"{catalog_config['title']} - HATS Pixel Density")
        plt.show()

    if catalog_config.get("plot_coverage", False) is not False:
        qa_log(f"[{catalog_config['title']}] Generating HATS coverage map")
        display_section_heading("HATS Coverage")
        coverage_kwargs = get_hats_plot_kwargs(catalog_config, "plot_coverage")
        fig, ax = lsdb_catalog.plot_coverage(**coverage_kwargs)
        ax.set_title(f"{catalog_config['title']} - HATS Coverage")
        plt.show()


def render_catalog_report(catalog_config, catalog_index):
    """Render all configured QA sections for one catalog."""

    catalog_title = catalog_config.get("title", f"Catalog {catalog_index}")
    qa_log(f"Starting catalog {catalog_index}: {catalog_title}")
    print(f"Processing catalog {catalog_index}: {catalog_title}")
    display(Markdown(f"## {catalog_title}"))

    if catalog_config.get("status", "available") == "planned":
        qa_log(f"[{catalog_title}] Rendering planned catalog placeholder")
        display(Markdown(str(catalog_config.get("markdown", "To be done"))))
        return

    catalog_context = render_catalog_basics(catalog_config)
    validate_hats_plot_sections(catalog_config, catalog_context)
    render_survey_area(catalog_config, catalog_context)
    render_unique_count(catalog_config, catalog_context)
    render_basic_statistics(catalog_config, catalog_context)
    render_spatial_distribution(catalog_config, catalog_context)
    render_magnitude_histograms(catalog_config, catalog_context)
    render_magnitude_statistics(catalog_config, catalog_context)
    render_magnitude_error_histograms(catalog_config, catalog_context)
    render_magnitude_error_statistics(catalog_config, catalog_context)
    render_magnitude_error_trends(catalog_config, catalog_context)
    render_hats_catalog_plots(catalog_config, catalog_context)
    del catalog_context


for catalog_index, catalog_config in enumerate(catalog_configs, start=1):
    render_catalog_report(catalog_config, catalog_index)


In [ ]:
client.close()
cluster.close()
